# Lesson 9 — Fine-tune the assembled ACT model

## The idea before the implementation

**Test whether fine-tuning improves ACT.** Fine-tuning continues training from saved weights. It can improve imitation on a target dataset, but it can also overfit or damage useful behavior. A successful optimizer call is therefore only the first part of an adaptation experiment.

**Where you work.** You work at the full training-and-evaluation level, using the assembled ACT model. Record a baseline, update on training episodes under a limit, save and reload, then compare held-out error and rollout behavior. The held-out rows do not update the model or fit normalization statistics.

**Pause and explain.** Explain why reloading the saved bundle is part of the experiment. The useful result must exist on disk with the processors needed to reproduce it.

Keep [the concept guide](../concepts/data-and-evaluation.md) open beside this notebook for the full system diagrams and lesson map.

You will update the assembled ACT model from Lesson 8 on the shared `bowl_plate` task. You will keep a fixed validation split, save a baseline and an adapted checkpoint, load the saved processors, and compare validation error and a LIBERO rollout.

This lesson assumes strong Python skills and basic machine learning. It explains the PyTorch calls used in the exercise. The real run uses the frozen local dataset. It does not download data.

## Model and training path

ACT maps the current state and two camera images to a future action chunk. For batch size $B$, chunk length $T=20$, and action width $D=7$, the predicted actions have shape $(B,T,D)$. `action_is_pad` has shape $(B,T)$; `True` marks action targets that do not exist because the episode ended. The state has shape $(B,8)$. Each image has shape $(B,3,H,W)$ after the saved processor, where $H$ and $W$ are the height and width in pixels.

Fine-tuning starts with parameter tensors $\theta$ from a saved ACT checkpoint. At update $k$, a gradient-descent step is:

$$\theta_{k+1}=\theta_k-\eta\,g_k, \qquad g_k=\nabla_{\theta}L(\theta_k)$$

Where:
* $\theta_k$ - all model parameters before update $k$.
* $\eta$ - the learning rate.
* $g_k$ - the gradient of the loss $L$ with respect to the parameters, on the current batch.

AdamW, used here, rescales $g_k$ per parameter and adds weight decay, but follows the same idea. The loss is masked action L1 plus a weighted latent KL term:

$$L=\frac{\sum_{b,t,d}v_{b,t}\left|\hat{a}_{b,t,d}-a_{b,t,d}\right|}{D\sum_{b,t}v_{b,t}}+\beta\,D_{\mathrm{KL}}\!\left(q(z\mid s,a)\,\middle\|\,\mathcal{N}(0,I)\right)$$

Where:
* $b$, $t$, $d$ - indices of the example, action step, and action value.
* $a_{b,t,d}$ - normalized target action; $\hat{a}_{b,t,d}$ - predicted action.
* $m_{b,t}$ - the padding mask `action_is_pad` (1 for padding); $v_{b,t}=1-m_{b,t}$ is 1 for a real action and 0 for padding.
* $q(z\mid s,a)$ - the posterior over the latent vector $z$, given state $s$ and target actions $a$.
* $D_{\mathrm{KL}}$ - the KL divergence from Lesson 4.
* $\beta$ - the KL weight `kl_weight`.

The mask applies to the action term. It prevents padded targets from changing either the numerator or denominator. The KL term is averaged over the batch and summed over latent dimensions by the supplied loss function.

The inputs and targets already pass through the saved ACT preprocessor. Image pixels are scaled from integer range 0–255 to float range 0–1; this is image scaling. State and action values use means and standard deviations saved from training episodes; this is feature standardization. These steps occur before model prediction and loss calculation. The postprocessor converts predicted normalized actions back to controller units. The environment then clips values to its valid command range. The validation split does not fit statistics and does not update parameters.

A small example: with two valid actions and one padded action, action errors $0.2$, $0.4$, and $9.0$ on one dimension give masked L1 $(0.2+0.4)/2=0.3$; the padded error $9.0$ has no effect. The action tensor still has all three time positions. Its mask decides which positions count.

<svg xmlns="http://www.w3.org/2000/svg" width="780" height="138" viewBox="0 0 780 138" role="img" aria-label="Adapt ACT and verify the saved result">
  <title>Adapt ACT and verify the saved result</title>
  <desc>Use the same processors and held-out rows for the before, after, and reload checks.</desc>
  <rect width="780" height="138" rx="8" fill="#f8fafc"/>
  <text x="12" y="22" font-family="sans-serif" font-size="15" font-weight="bold" fill="#0f172a">Adapt ACT and verify the saved result</text>
  <rect x="12" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="97.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Fixed baseline</text>
  <text x="97.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">measurements</text>
  <path d="M 186 67 H 203 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="207" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="292.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Train-split loss</text>
  <text x="292.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">and updates</text>
  <path d="M 381 67 H 398 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="402" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="487.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Save weights</text>
  <text x="487.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">and processors</text>
  <path d="M 576 67 H 593 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="597" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="682.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Reload and</text>
  <text x="682.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">compare</text>
  <text x="12" y="120" font-family="sans-serif" font-size="12" fill="#334155">Use the same processors and held-out rows for the before, after, and reload checks.</text>
</svg>

Training rows pass through the saved preprocessor into ACT and its masked loss. Only training updates change weights. Validation rows measure error without updates. Evaluate the starting and reloaded policies in LIBERO to measure behavior as well.


The training data uses 10 frames per second as its storage label. LIBERO control runs at 20 steps per second. Each relative action is one environment step. The dataset storage rate does not set an action repeat rule. We do not decimate actions here. A 200-step rollout therefore has at most 10 seconds of simulated control time, apart from reset time.

## Exercise: implement a bounded update path

Implement `fine_tune_updates`. Each update must transfer only the current batch to the selected device, call the ACT loss, backpropagate, clip the gradient norm, and update the model. `steps` is an update count. `max_seconds` is a wall-clock budget checked by the supplied training loop before each update. `batches` is a sequence of ACT batches (`dict[str, Tensor]` keyed by LeRobot feature names, as in Lesson 3). The function returns a `list[dict[str, float]]`: one metric dictionary for each update that completed before that limit. The fixed CPU fixture below checks parameter change and reduced action error. The required experiment uses this same function with the assembled ACT model and real data.

In [1]:
import json
import os
import sys
import time
from collections import deque
from collections.abc import Mapping, Sequence
from dataclasses import asdict
from pathlib import Path
from typing import cast
from uuid import uuid4

import numpy as np
import torch
from torch import Tensor, nn


def find_workspace(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "manifests" / "act_references.yaml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the VLA course workspace.")


WORKSPACE = find_workspace()
sys.path.insert(0, str(WORKSPACE / "src"))
HF_ROOT = WORKSPACE / ".local-data" / "hf"
os.environ["HF_HUB_CACHE"] = str(HF_ROOT / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_ROOT / "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Workspace:", WORKSPACE)
print("PyTorch:", torch.__version__)

Workspace: /home/michal/code/vlas_tutorial
PyTorch: 2.11.0+cu128


In [3]:
from lerobot.policies.act import ACTConfig, ACTPolicy
from lerobot.processor import PolicyProcessorPipeline, load_pretrained_policy_processors

from vla_lab.evaluation.demo import encode_rgb_video
from vla_lab.policy.act import make_act_processors
from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.act_model import ACTModel, act_from_lerobot
from vla_lab.teaching.interfaces import ModelT
from vla_lab.teaching.live import load_act_batches
from vla_lab.teaching.training import masked_action_l1, diagonal_gaussian_kl

## Exercise 1 — Test the bounded update path

This small check uses fixed CPU tensors. It does not need model weights, a simulator, or local episodes. Implement the function in the next cell, then use these checks to confirm that it changes model parameters and reduces error on valid actions.

More behaviour checks for this exercise: `check_fine_tune_updates` in [lesson09_checks.py](lesson09_checks.py).

In [6]:
def fine_tune_updates(
    model: ModelT,
    batches: Sequence[Mapping[str, Tensor]],
    optimizer: torch.optim.Optimizer,
    *,
    steps: int,
    device: torch.device,
    kl_weight: float,
    max_grad_norm: float,
    max_seconds: float | None = None,
) -> list[dict[str, float]]:
    """Run bounded ACT updates and move each batch only when it is used."""
    batches = list(batches)

    if len(batches) == 0:
        raise ValueError("No batches provided")
    if steps < 0:
        raise ValueError("Number of steps must be nonnegative")
    if max_seconds is not None and max_seconds <= 0:
        raise ValueError("Time limit must be positive")

    start = time.time()
    infos = []
    model.train()

    def loss_fn(batch: Mapping[str, torch.Tensor]):
        predicted, (mu, logvar) = model(batch)
        action_loss = masked_action_l1(predicted, batch["action"], batch["action_is_pad"])
        kl = diagonal_gaussian_kl(mu, logvar)
        return action_loss + kl_weight * kl, {"action_l1": action_loss, "kl": kl}

    for step in range(steps):
        batch = {key: value.to(device) for key, value in batches[step % len(batches)].items()}
        optimizer.zero_grad()

        loss, info = loss_fn(batch)
        if loss.ndim != 0 or not torch.isfinite(loss):
            raise ValueError("loss_fn must return one finite scalar tensor")
        loss.backward()

        if max_grad_norm is not None:
            if max_grad_norm <= 0:
                raise ValueError("max_grad_norm must be positive")
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        info = {key: float(value) for key, value in info.items()}
        info["loss"] = float(loss)
        infos.append(info)

        if max_seconds is not None and time.time() - start > max_seconds:
            break

    return infos


class MiniACT(nn.Module):
    """A small predictor for the update check; it has the same output contract."""

    def __init__(self) -> None:
        super().__init__()
        self.action_bias = nn.Parameter(torch.zeros(2))

    def forward(self, batch: Mapping[str, Tensor]) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        target = batch["action"]
        prediction = self.action_bias.view(1, 1, 2).expand_as(target)
        zero = torch.zeros((target.shape[0], 1), device=target.device)
        return prediction, (zero, zero)


mini_batch = {
    "action": torch.tensor(
        [[[0.4, -0.2], [0.4, -0.2], [0.4, -0.2]], [[0.4, -0.2], [0.4, -0.2], [0.4, -0.2]]]
    ),
    "action_is_pad": torch.tensor([[False, False, False], [False, False, True]]),
}
mini_model = MiniACT()
mini_optimizer = torch.optim.AdamW(mini_model.parameters(), lr=0.05)
with torch.inference_mode():
    before = masked_action_l1(
        mini_model(mini_batch)[0], mini_batch["action"], mini_batch["action_is_pad"]
    )
mini_history = fine_tune_updates(
    mini_model,
    [mini_batch],
    mini_optimizer,
    steps=8,
    device=torch.device("cpu"),
    kl_weight=0.01,
    max_grad_norm=1.0,
)
with torch.inference_mode():
    after = masked_action_l1(
        mini_model(mini_batch)[0], mini_batch["action"], mini_batch["action_is_pad"]
    )
assert len(mini_history) == 8, "Return one metric row for each completed update."
assert after < before, "Repeated updates must reduce error on the fixed valid targets."
print(
    {
        "updates": len(mini_history),
        "valid_action_l1_before": float(before),
        "valid_action_l1_after": float(after),
    }
)

# More behaviour checks: lesson09_checks.py, next to this notebook.
from lesson09_checks import check_fine_tune_updates

check_fine_tune_updates(fine_tune_updates)

{'updates': 8, 'valid_action_l1_before': 0.30000001192092896, 'valid_action_l1_after': 0.047347746789455414}
✓ loss = masked L1 + kl_weight * KL; batches cycle in order; inputs unchanged
✓ kl_weight scales the KL gradient
✓ max_grad_norm clips the total gradient norm
✓ steps = 0 runs no update
✓ max_seconds stops the loop early and keeps the completed rows


## Full experiment — prepare a baseline

The real run uses 256 training batches and 64 fixed validation batches. It starts from the prepared `bowl_plate` ACT checkpoint and copies its weights into the assembled `ACTModel` from Lesson 8. It saves the model state, model settings, optimizer state, frozen data identity, and both processor pipelines before the first update. The baseline validation and rollout use the processors loaded back from that saved checkpoint.

In [7]:
TASK_ALIAS = "bowl_plate"
SOURCE_CHECKPOINT = WORKSPACE / ".local-data/runs/act-bowl-plate-probe-200/checkpoint-000200"
CHUNK_SIZE = 20
BATCH_SIZE = 8
TRAIN_BATCH_COUNT = 256
VALIDATION_BATCH_COUNT = 64
UPDATE_BUDGET = 1500
UPDATE_BLOCK = 100
WALL_LIMIT_S = 2 * 60 * 60
SAVE_RESERVE_S = 180
SEED = 29
KL_WEIGHT = 0.01
LEARNING_RATE = 1e-5
MAX_GRAD_NORM = 1.0
RUN_STARTED = time.monotonic()

if not torch.cuda.is_available():
    raise RuntimeError("The required full adaptation experiment needs CUDA.")
GPU_BYTES = torch.cuda.get_device_properties(0).total_memory
if GPU_BYTES > 48 * 1024**3:
    torch.cuda.set_per_process_memory_fraction((48 * 1024**3) / GPU_BYTES)
DEVICE = torch.device("cuda")
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.cuda.reset_peak_memory_stats(DEVICE)
OUTPUT = WORKSPACE / ".local-data/course-runs/unit09" / uuid4().hex
OUTPUT.mkdir(parents=True, exist_ok=False)

DATA = load_act_batches(
    TASK_ALIAS,
    WORKSPACE,
    batch_size=BATCH_SIZE,
    chunk_size=CHUNK_SIZE,
    train_batches=TRAIN_BATCH_COUNT,
    validation_batches=VALIDATION_BATCH_COUNT,
    seed=SEED,
)
print("Frozen train/validation identity:", DATA.metadata)
STARTER_POLICY = ACTPolicy.from_pretrained(SOURCE_CHECKPOINT, local_files_only=True)
STARTER_POLICY.config = cast(ACTConfig, STARTER_POLICY.config)
STARTER_POLICY.config = cast(ACTConfig, STARTER_POLICY.config)
PREPROCESSOR, POSTPROCESSOR = make_act_processors(
    STARTER_POLICY.config, TASK_ALIAS, WORKSPACE / "configs"
)
MODEL = act_from_lerobot(STARTER_POLICY.model).to(DEVICE)
OPTIMIZER = torch.optim.AdamW(MODEL.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)


def validation_action_l1(
    model: ACTModel,
    batches: Sequence[Mapping[str, Tensor]],
    device: torch.device,
) -> float:
    model.eval()
    values = []
    with torch.inference_mode():
        for cpu_batch in batches:
            batch = {name: value.to(device) for name, value in cpu_batch.items()}
            prediction, _latent = model(batch)
            values.append(masked_action_l1(prediction, batch["action"], batch["action_is_pad"]))
    return float(torch.stack(values).mean().cpu())


class AssembledACTPolicy:
    """Send the predicted chunk one action at a time to the episode runner."""

    def __init__(self, model: ACTModel) -> None:
        self.model = model
        self.actions: deque[Tensor] = deque()

    def reset(self) -> None:
        self.actions.clear()

    def select_action(self, batch: dict[str, Tensor]) -> Tensor:
        if not self.actions:
            chunk, _latent = self.model(batch)
            self.actions.extend(chunk[0, index] for index in range(CHUNK_SIZE))
        return self.actions.popleft().unsqueeze(0)


def save_assembled_checkpoint(
    model: ACTModel,
    optimizer: torch.optim.Optimizer,
    preprocessor: PolicyProcessorPipeline,
    postprocessor: PolicyProcessorPipeline,
    destination: Path,
    *,
    updates: int,
    status: str,
) -> Path:
    destination.mkdir(parents=True, exist_ok=False)
    torch.save(
        {"model": model.state_dict(), "optimizer": optimizer.state_dict()}, destination / "state.pt"
    )
    preprocessor.save_pretrained(destination)
    postprocessor.save_pretrained(destination)
    manifest = {
        "schema_version": 1,
        "model": "assembled-act",
        "task_alias": TASK_ALIAS,
        "source_checkpoint": str(SOURCE_CHECKPOINT),
        "updates": updates,
        "status": status,
        "act_settings": asdict(model.settings),
        "seed": SEED,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-4,
        "kl_weight": KL_WEIGHT,
        "max_grad_norm": MAX_GRAD_NORM,
        "optimizer": "AdamW",
        "dataset_identity": DATA.metadata,
    }
    (destination / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n"
    )
    return destination


def load_assembled_checkpoint(
    checkpoint: Path,
    device: torch.device,
) -> tuple[ACTModel, PolicyProcessorPipeline, PolicyProcessorPipeline, dict[str, object]]:
    manifest: dict[str, object] = json.loads((checkpoint / "manifest.json").read_text())
    source = ACTPolicy.from_pretrained(SOURCE_CHECKPOINT, local_files_only=True)
    model = act_from_lerobot(source.model)
    payload = torch.load(checkpoint / "state.pt", map_location=device, weights_only=True)
    model.load_state_dict(payload["model"], strict=True)
    model = model.to(device).eval()
    preprocessor, postprocessor = load_pretrained_policy_processors(checkpoint)
    return model, preprocessor, postprocessor, manifest


def run_assembled_rollout(
    model: ACTModel,
    preprocessor: PolicyProcessorPipeline,
    postprocessor: PolicyProcessorPipeline,
    output_dir: Path,
) -> dict[str, object]:
    output_dir.mkdir(parents=True, exist_ok=False)
    environment = open_libero_task(
        TASK_ALIAS,
        0,
        WORKSPACE / "configs",
        WORKSPACE / ".local-data",
        control_hz=20,
    )
    frames: list[np.ndarray] = []
    try:
        result = run_policy_episode(
            environment,
            AssembledACTPolicy(model.eval()),
            preprocessor,
            postprocessor,
            TASK_ALIAS,
            0,
            WORKSPACE / "configs",
            partition="development",
            max_steps=200,
            on_frame=frames.append,
            reset_seed=20261006,
            control_hz=20,
        )
    finally:
        environment.close()
    video = encode_rgb_video(frames, output_dir / "rollout.mp4", fps=20)
    return {
        "success": result.success,
        "steps": result.steps,
        "termination_reason": result.termination_reason,
        "control_hz": result.control_hz,
        "simulated_duration_s": result.simulated_duration_s,
        "video": str(video),
    }


BASELINE_DIR = save_assembled_checkpoint(
    MODEL,
    OPTIMIZER,
    PREPROCESSOR,
    POSTPROCESSOR,
    OUTPUT / "baseline",
    updates=0,
    status="baseline",
)
BASELINE_MODEL, BASELINE_PREPROCESSOR, BASELINE_POSTPROCESSOR, BASELINE_MANIFEST = (
    load_assembled_checkpoint(
        BASELINE_DIR,
        DEVICE,
    )
)
BEFORE_VALIDATION_L1 = validation_action_l1(BASELINE_MODEL, DATA.validation, DEVICE)
ROLLOUT_BEFORE = run_assembled_rollout(
    BASELINE_MODEL,
    BASELINE_PREPROCESSOR,
    BASELINE_POSTPROCESSOR,
    OUTPUT / "rollout-before",
)
print("Baseline:", {"validation_action_l1": BEFORE_VALIDATION_L1, "rollout": ROLLOUT_BEFORE})

Frozen train/validation identity: {'task_alias': 'bowl_plate', 'dataset_revision': 'a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4', 'split_sha256': 'ef237195fc6ace8a78ff69fcf472362a89adb64a4ff079b5fb84724157bb82f4', 'train_episode_ids': [379, 422, 426, 431, 433, 447, 448, 451, 459, 466, 481, 483, 488, 507, 511, 513, 522, 532, 537, 549, 551, 563, 568, 582, 607, 615, 620, 621, 626, 634, 639, 642, 646, 653, 655, 670, 679, 708, 716], 'validation_episode_ids': [718, 726, 727, 749, 750, 768, 770, 801, 803, 806], 'seed': 29, 'chunk_size': 20, 'batch_size': 8, 'train_rows': 2048, 'validation_rows': 512, 'normalization_source': '/home/michal/code/vlas_tutorial/configs/data/stats/bowl_plate.json', 'device': 'cpu'}
Loading weights from local directory
Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Baseline: {'validation_action_l1': 0.4941685199737549, 'rollout': {'success': True, 'steps': 71, 'termination_reason': 'success', 'control_hz': 20, 'simulated_du

## Full experiment — fine-tune on training episodes

Each block uses the learner function from the exercise. The validation batches stay outside the optimizer. The deadline includes setup time and keeps three minutes for checkpoint save and evaluation. The training loop checks its time budget before each update. It may finish the update that began just before the deadline. Each completed block writes a checkpoint with its optimizer state and update count.

In [8]:
DEADLINE = RUN_STARTED + WALL_LIMIT_S - SAVE_RESERVE_S
updates = 0
history: list[dict[str, float]] = []
LAST_CHECKPOINT: Path | None = None
while updates < UPDATE_BUDGET and time.monotonic() < DEADLINE:
    remaining_s = DEADLINE - time.monotonic()
    if remaining_s <= 0:
        break
    requested = min(UPDATE_BLOCK, UPDATE_BUDGET - updates)
    # Continue through all batches when a checkpoint block ends.
    batch_offset = updates % len(DATA.train)
    block_batches = DATA.train[batch_offset:] + DATA.train[:batch_offset]
    block_history = fine_tune_updates(
        MODEL,
        block_batches,
        OPTIMIZER,
        steps=requested,
        device=DEVICE,
        kl_weight=KL_WEIGHT,
        max_grad_norm=MAX_GRAD_NORM,
        max_seconds=remaining_s,
    )
    history.extend(block_history)
    updates += len(block_history)
    status = "complete" if updates == UPDATE_BUDGET else "incomplete"
    LAST_CHECKPOINT = save_assembled_checkpoint(
        MODEL,
        OPTIMIZER,
        PREPROCESSOR,
        POSTPROCESSOR,
        OUTPUT / f"checkpoint-update-{updates:05d}",
        updates=updates,
        status=status,
    )
    if block_history:
        print(
            f"updates={updates}/{UPDATE_BUDGET} elapsed_s={time.monotonic() - RUN_STARTED:.1f} loss={block_history[-1]['loss']:.5f}"
        )
    if len(block_history) < requested:
        break

TRAINING_COMPLETE = updates == UPDATE_BUDGET
if LAST_CHECKPOINT is None:
    LAST_CHECKPOINT = save_assembled_checkpoint(
        MODEL,
        OPTIMIZER,
        PREPROCESSOR,
        POSTPROCESSOR,
        OUTPUT / "checkpoint-update-00000",
        updates=0,
        status="incomplete",
    )
print(
    {
        "training_complete": TRAINING_COMPLETE,
        "updates": updates,
        "update_budget": UPDATE_BUDGET,
        "checkpoint": str(LAST_CHECKPOINT),
    }
)

updates=100/1500 elapsed_s=70.6 loss=0.54567
updates=200/1500 elapsed_s=75.8 loss=0.46282
updates=300/1500 elapsed_s=81.0 loss=0.51755
updates=400/1500 elapsed_s=86.1 loss=0.53833
updates=500/1500 elapsed_s=91.4 loss=0.49655
updates=600/1500 elapsed_s=96.7 loss=0.57442
updates=700/1500 elapsed_s=101.9 loss=0.50230
updates=800/1500 elapsed_s=107.1 loss=0.46002
updates=900/1500 elapsed_s=112.4 loss=0.52890
updates=1000/1500 elapsed_s=118.0 loss=0.44572
updates=1100/1500 elapsed_s=123.6 loss=0.46109
updates=1200/1500 elapsed_s=129.3 loss=0.51915
updates=1300/1500 elapsed_s=134.7 loss=0.49102
updates=1400/1500 elapsed_s=140.2 loss=0.37118
updates=1500/1500 elapsed_s=145.7 loss=0.41316
{'training_complete': True, 'updates': 1500, 'update_budget': 1500, 'checkpoint': '/home/michal/code/vlas_tutorial/.local-data/course-runs/unit09/7832c8610c0a43ca87aeb5309f5dec30/checkpoint-update-01500'}


## Full experiment — reload and compare

Load the last saved model and both saved processor pipelines. Compare it with the baseline on the same held-out validation batches and the same development state. The record keeps validation action error separate from LIBERO success. A partial run stays marked incomplete.

In [9]:
if LAST_CHECKPOINT is None:
    raise RuntimeError("The training stage did not save a checkpoint.")
FINAL_CHECKPOINT: Path = LAST_CHECKPOINT
FINAL_MODEL, FINAL_PREPROCESSOR, FINAL_POSTPROCESSOR, FINAL_MANIFEST = load_assembled_checkpoint(
    FINAL_CHECKPOINT,
    DEVICE,
)
AFTER_VALIDATION_L1 = validation_action_l1(FINAL_MODEL, DATA.validation, DEVICE)
ROLLOUT_AFTER = run_assembled_rollout(
    FINAL_MODEL,
    FINAL_PREPROCESSOR,
    FINAL_POSTPROCESSOR,
    OUTPUT / "rollout-after",
)
record = {
    "status": "complete" if TRAINING_COMPLETE else "incomplete",
    "task_alias": TASK_ALIAS,
    "updates": updates,
    "update_budget": UPDATE_BUDGET,
    "elapsed_s": time.monotonic() - RUN_STARTED,
    "device": str(DEVICE),
    "gpu_peak_memory_gb": torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
    "dataset_identity": DATA.metadata,
    "validation_action_l1_before": BEFORE_VALIDATION_L1,
    "validation_action_l1_after": AFTER_VALIDATION_L1,
    "validation_action_l1_change": AFTER_VALIDATION_L1 - BEFORE_VALIDATION_L1,
    "baseline_manifest": BASELINE_MANIFEST,
    "final_manifest": FINAL_MANIFEST,
    "rollout_before": ROLLOUT_BEFORE,
    "rollout_after": ROLLOUT_AFTER,
    "checkpoint": str(LAST_CHECKPOINT),
}
(OUTPUT / "adaptation_record.json").write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
print(json.dumps(record, indent=2, sort_keys=True))

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{
  "baseline_manifest": {
    "act_settings": {
      "action_dim": 7,
      "activation": "relu",
      "chunk_size": 20,
      "dim_feedforward": 1024,
      "dim_model": 256,
      "dropout": 0.1,
      "image_keys": [
        "observation.images.image",
        "observation.images.image2"
      ],
      "latent_dim": 32,
      "n_decoder_layers": 1,
      "n_encoder_layers": 2,
      "n_heads": 4,
      "n_vae_encoder_layers": 2,
      "pre_norm": false,
      "state_dim": 8,
      "vision_backbone": "resnet18"
    },
    "dataset_identity": {
      "batch_size": 8,
      "chunk_size": 20,
      "dataset_revision": "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4",
      "device": "cpu",
      "normalization_source": "/home/michal/code/vlas_tutorial/configs/data/stats/bowl_plate.json",
      "seed": 29,
      "split_sha256": "ef237195fc6ace8a78ff69fcf472362a89adb64a4ff079b5fb84724157bb82f4",
    

## Limits

A lower validation error does not prove task success. One development rollout does not prove performance on all states. The run reports the checkpoint, processors, episode split, validation error, rollout result, simulated control time, elapsed wall time, and peak GPU memory.